In [1]:
from sqlalchemy import create_engine, Integer, String, Date, VARCHAR, text
from datetime import datetime
import pandas as pd

In [3]:
pd.set_option('display.max_columns', None)
caminho_arquivo = r"C:\Users\João\Desktop\postgre\origem_dados\V_OCORRENCIA_AMPLA.json"
df = pd.read_json(caminho_arquivo, encoding='utf-8-sig')
colunas = ["Numero_da_Ocorrencia", "Classificacao_da_Ocorrência", "Data_da_Ocorrencia","Municipio","UF","Regiao","Nome_do_Fabricante"]
df = df[colunas]
df.rename(columns={'Classificacao_da_Ocorrência' : 'Classificacao_da_Ocorrencia'} ,inplace=True)

In [4]:
# Convertendo coluna em datas
df['Data_da_Ocorrencia'] = pd.to_datetime(df['Data_da_Ocorrencia'])

In [5]:
ano_aula = datetime.now().year - 2
df = df[df['Data_da_Ocorrencia'].dt.year == ano_aula]

In [6]:
# Conectando com o banco de dados
dbname = 'python'
user = 'postgres'
password = '56789'
host = 'localhost'
port = '5432' 

conexao_str = f'postgresql://{user}:{password}@{host}:{port}/{dbname}'
engine = create_engine(conexao_str)

nome_tabela = 'anac_sqlalchemy'

# Deletando registros com base no ano atual
cursor = engine.connect()
delete = text(f'DELETE FROM {nome_tabela} WHERE EXTRACT(YEAR FROM "Data_da_Ocorrencia") = {ano_aula}')
cursor.execute(delete)
cursor.commit()

# Enviando o dataframe para o banco de dados
df.to_sql(nome_tabela, engine, index=False, if_exists='append',
dtype={
	'Numero_da_Ocorrencia' : Integer,
	'Classificacao_da_Ocorrencia' : VARCHAR(30),
	'Data_da_Ocorrencia' : Date
})

engine.dispose()
cursor.close()